# Entropie, Cross-Entropy & KL-Divergenz

Diese drei Konzepte bilden das informationstheoretische Fundament des Machine Learning. Sie messen **Unsicherheit**, **Unterschiede zwischen Verteilungen** und sind die mathematische Grundlage für Loss-Funktionen in der Klassifikation.

In diesem Notebook:
1. **Shannon-Entropie** – wie viel Information steckt in einer Verteilung?
2. **Cross-Entropy** – der wichtigste Loss für Klassifikation
3. **KL-Divergenz** – wie unterschiedlich sind zwei Verteilungen?
4. **Softmax** – von Logits zu Wahrscheinlichkeiten

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['figure.dpi'] = 100

## 1. Die Algorithmen

In [ ]:
def softmax(x: np.ndarray) -> np.ndarray:
    """Softmax mit numerischer Stabilität."""
    shifted = x - np.max(x, axis=-1, keepdims=True)
    exp = np.exp(shifted)
    return exp / np.sum(exp, axis=-1, keepdims=True)


def sigmoid(x: np.ndarray) -> np.ndarray:
    """Sigmoid: 1/(1+e^(-x))"""
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))


def entropy(p: np.ndarray) -> float:
    """Shannon-Entropie: H = -Σ p_i * log₂(p_i)"""
    p = np.clip(p, 1e-10, 1)
    return -np.sum(p * np.log2(p))


def cross_entropy(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Cross-Entropy Loss (binär)."""
    y_pred = np.clip(y_pred, 1e-10, 1 - 1e-10)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))


def kl_divergence(p: np.ndarray, q: np.ndarray) -> float:
    """Kullback-Leibler-Divergenz: D_KL(P||Q) in Bits (log₂)."""
    p = np.clip(p, 1e-10, 1)
    q = np.clip(q, 1e-10, 1)
    return np.sum(p * np.log2(p / q))

## 2. Shannon-Entropie

Die Entropie $H$ misst die **Unsicherheit** einer Wahrscheinlichkeitsverteilung:

$$H(P) = -\sum_{i} p_i \cdot \log_2(p_i)$$

- **Gleichverteilung** → maximale Entropie (maximale Unsicherheit)
- **Sichere Verteilung** (eine Klasse $p_i \approx 1$) → Entropie nahe 0

In [ ]:
# Entropie-Beispiele
p_uniform = np.array([0.25, 0.25, 0.25, 0.25])       # 4 Klassen, gleichverteilt
p_skewed  = np.array([0.70, 0.15, 0.10, 0.05])       # schief
p_certain = np.array([0.99, 0.01])                    # fast sicher
p_perfect = np.array([1.0, 0.0])                      # perfekt sicher

print(f"Entropie (gleichverteilt, 4 Klassen): {entropy(p_uniform):.4f} bit  (max: {np.log2(4):.4f})")
print(f"Entropie (schief):                     {entropy(p_skewed):.4f} bit")
print(f"Entropie (fast sicher):                {entropy(p_certain):.4f} bit")
print(f"Entropie (perfekt sicher):             {entropy(p_perfect):.4f} bit")

In [ ]:
# Visualisierung: Entropie einer Münze (Bernoulli-Verteilung)
p_heads = np.linspace(0.001, 0.999, 200)
entropies = [entropy(np.array([p, 1-p])) for p in p_heads]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(p_heads, entropies, 'b-', linewidth=2.5)
ax.axvline(x=0.5, color='red', linestyle='--', alpha=0.5, label='p=0.5 (max. Entropie)')
ax.fill_between(p_heads, entropies, alpha=0.15, color='blue')
ax.set_xlabel('p(Kopf)')
ax.set_ylabel('Entropie (bit)')
ax.set_title('Entropie einer Münze: H = -p·log₂(p) - (1-p)·log₂(1-p)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Cross-Entropy

Die Cross-Entropy misst, wie gut eine vorhergesagte Verteilung $\hat{y}$ zur wahren Verteilung $y$ passt:

$$H(y, \hat{y}) = -\frac{1}{N}\sum_{i} \left[ y_i \log(\hat{y}_i) + (1-y_i) \log(1-\hat{y}_i) \right]$$

Das ist der **Binary Cross-Entropy Loss** – die Standard-Loss-Funktion für binäre Klassifikation.

In [ ]:
# Cross-Entropy: je besser die Vorhersage, desto kleiner der Loss
y_true = np.array([1.0, 0.0, 1.0, 1.0, 0.0])  # wahre Labels

# Drei verschiedene Vorhersagen
y_good  = np.array([0.95, 0.05, 0.90, 0.85, 0.10])   # gute Vorhersage
y_ok    = np.array([0.70, 0.30, 0.65, 0.60, 0.40])   # mittelmäßig
y_bad   = np.array([0.10, 0.90, 0.20, 0.15, 0.80])   # schlecht (fast invers)

print(f"Cross-Entropy (gute Vorhersage):     {cross_entropy(y_true, y_good):.4f}")
print(f"Cross-Entropy (mittelmäßig):         {cross_entropy(y_true, y_ok):.4f}")
print(f"Cross-Entropy (schlechte Vorhersage): {cross_entropy(y_true, y_bad):.4f}")

In [ ]:
# Visualisierung: Cross-Entropy für eine einzelne Vorhersage
# Loss = -[y·log(ŷ) + (1-y)·log(1-ŷ)]
y_hat = np.linspace(0.001, 0.999, 200)

fig, ax = plt.subplots(figsize=(10, 5))
loss_y1 = -np.log(y_hat)           # wenn y_true = 1
loss_y0 = -np.log(1 - y_hat)       # wenn y_true = 0

ax.plot(y_hat, loss_y1, 'b-', linewidth=2, label='Loss wenn y=1: -log(ŷ)')
ax.plot(y_hat, loss_y0, 'r-', linewidth=2, label='Loss wenn y=0: -log(1-ŷ)')
ax.axvline(x=0.5, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Vorhersage ŷ')
ax.set_ylabel('Loss')
ax.set_title('Binary Cross-Entropy Loss pro Datenpunkt')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 5)
plt.tight_layout()
plt.show()

## 4. KL-Divergenz

Die Kullback-Leibler-Divergenz misst, wie sehr sich eine Verteilung $Q$ von einer Referenzverteilung $P$ unterscheidet:

$$D_{KL}(P \parallel Q) = \sum_i p_i \cdot \log_2\left(\frac{p_i}{q_i}\right)$$

- **$D_{KL} = 0$** wenn $P = Q$ (identische Verteilungen)
- **$D_{KL} > 0$** sonst – je größer, desto unterschiedlicher
- **Nicht symmetrisch:** $D_{KL}(P \parallel Q) \neq D_{KL}(Q \parallel P)$

In [ ]:
# KL-Divergenz: P = Referenz, Q = Approximation
P = np.array([0.5, 0.3, 0.2])          # wahre Verteilung
Q1 = np.array([0.5, 0.3, 0.2])         # identisch
Q2 = np.array([0.4, 0.4, 0.2])         # leicht anders
Q3 = np.array([0.1, 0.1, 0.8])         # stark anders
Q4 = np.array([0.33, 0.33, 0.34])      # gleichverteilt

print(f"D_KL(P || Q1) identisch:       {kl_divergence(P, Q1):.6f} bit")
print(f"D_KL(P || Q2) leicht anders:   {kl_divergence(P, Q2):.4f} bit")
print(f"D_KL(P || Q3) stark anders:    {kl_divergence(P, Q3):.4f} bit")
print(f"D_KL(P || Q4) gleichverteilt:  {kl_divergence(P, Q4):.4f} bit")
print()
print("Asymmetrie:")
print(f"  D_KL(P || Q3) = {kl_divergence(P, Q3):.4f}")
print(f"  D_KL(Q3 || P) = {kl_divergence(Q3, P):.4f}  ← anderer Wert!")

### Zusammenhang: Cross-Entropy = Entropie + KL-Divergenz

$$H(P, Q) = H(P) + D_{KL}(P \parallel Q)$$

Das erklärt, warum Cross-Entropy als Loss funktioniert: $H(P)$ ist konstant (die wahre Verteilung ändert sich nicht), also ist das Minimieren der Cross-Entropy äquivalent zum Minimieren der KL-Divergenz.

In [ ]:
# Verifikation: H(P,Q) = H(P) + D_KL(P||Q)
P = np.array([0.7, 0.2, 0.1])
Q = np.array([0.5, 0.3, 0.2])

ce = cross_entropy(P, Q)  # Achtung: unsere cross_entropy ist binär!
# Für kategoriale Cross-Entropy:
ce_cat = -np.sum(P * np.log2(np.clip(Q, 1e-10, 1)))
h_p = entropy(P)
kl = kl_divergence(P, Q)

print(f"H(P)           = {h_p:.4f}")
print(f"D_KL(P||Q)     = {kl:.4f}")
print(f"H(P) + D_KL    = {h_p + kl:.4f}")
print(f"H(P, Q) direkt = {ce_cat:.4f}")
print(f"Differenz:       {abs(ce_cat - (h_p + kl)):.10f} ✓")

## 5. Softmax – Von Logits zu Wahrscheinlichkeiten

Softmax wandelt beliebige reelle Zahlen (Logits) in eine Wahrscheinlichkeitsverteilung um:

$$\text{softmax}(x_i) = \frac{e^{x_i}}{\sum_j e^{x_j}}$$

Der **numerische Trick** ($x_i - \max(x)$) verhindert Overflow bei großen Werten.

In [ ]:
# Softmax in Aktion
logits = np.array([2.0, 1.0, 0.1])
probs = softmax(logits)

print("Logits: ", logits)
print("Softmax:", probs)
print(f"Summe:    {probs.sum():.6f} (muss 1.0 sein)")
print()

# Temperatur-Effekt
print("Temperatur-Skalierung (je kleiner T, desto 'härter' die Verteilung):")
for T in [0.5, 1.0, 2.0, 5.0]:
    probs_t = softmax(logits / T)
    print(f"  T={T:.1f}: {probs_t}")

In [ ]:
# Visualisierung: Softmax-Temperatur
logits_demo = np.array([3.0, 1.0, 0.5, 0.2])
temperatures = [0.2, 0.5, 1.0, 2.0, 5.0]
labels = ['Klasse A', 'Klasse B', 'Klasse C', 'Klasse D']

fig, axes = plt.subplots(1, len(temperatures), figsize=(15, 4))

for ax, T in zip(axes, temperatures):
    probs = softmax(logits_demo / T)
    colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(probs)))
    bars = ax.bar(labels, probs, color=colors, edgecolor='white')
    ax.set_ylim(0, 1)
    ax.set_title(f'T = {T}')
    ax.set_ylabel('Wahrscheinlichkeit')
    # Werte auf Balken
    for bar, val in zip(bars, probs):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.2f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Softmax mit Temperatur-Skalierung', fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

## 6. Zusammenfassung

| Konzept | Formel | Bedeutung |
|---------|--------|-----------|
| **Entropie $H(P)$** | $-\sum p_i \log_2 p_i$ | Unsicherheit einer Verteilung |
| **Cross-Entropy $H(P,Q)$** | $-\sum p_i \log_2 q_i$ | Wie gut approximiert $Q$ die wahre Verteilung $P$? |
| **KL-Divergenz $D_{KL}$** | $\sum p_i \log_2(p_i/q_i)$ | „Abstand" zwischen zwei Verteilungen (asymmetrisch!) |
| **Softmax** | $e^{x_i} / \sum e^{x_j}$ | Logits → Wahrscheinlichkeiten |

**Merksatz:** $H(P,Q) = H(P) + D_{KL}(P \parallel Q)$ – Cross-Entropy minimieren = KL-Divergenz minimieren.